# 🚚 Delivery Route Optimization — ADA Capstone Project
**Author:** Arpita  
**Repo:** `delivery-route-mini-project-Arpita`  
**Problem:** E-commerce last-mile delivery optimization using Recurrence, Greedy, DP, Dijkstra, MST, and TSP

---

## Task 1 — Environment Setup

In [ ]:
# Install required packages (run once)
# !pip install matplotlib memory_profiler networkx jupyterlab

import itertools
import heapq
import time
import random
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from memory_profiler import memory_usage

print("✅ All libraries imported successfully.")

## Task 2 — Input Modeling & Constraint Setup

We define:
- **8 delivery locations** (0 = Warehouse/Depot)
- A **symmetric distance matrix** between all locations
- **Parcel metadata**: value, weight, time-window [earliest, latest delivery hour]
- **Vehicle constraint**: max weight capacity

In [ ]:
# ─────────────────────────────────────────────
# LOCATIONS
# ─────────────────────────────────────────────
LOCATIONS = [
    "Warehouse",   # 0 — depot
    "Sector A",    # 1
    "Sector B",    # 2
    "Sector C",    # 3
    "Sector D",    # 4
    "Sector E",    # 5
    "Sector F",    # 6
    "Sector G",    # 7
]

N = len(LOCATIONS)  # 8 nodes

# ─────────────────────────────────────────────
# DISTANCE MATRIX  (symmetric, in km)
# dist[i][j] = distance from location i to j
# ─────────────────────────────────────────────
DIST = [
    #  0    1    2    3    4    5    6    7
    [  0,  10,  15,  20,  12,  18,  25,  30],  # 0 Warehouse
    [ 10,   0,  12,  15,  20,  22,  18,  25],  # 1 Sector A
    [ 15,  12,   0,   8,  16,  14,  10,  20],  # 2 Sector B
    [ 20,  15,   8,   0,  10,   6,  12,  18],  # 3 Sector C
    [ 12,  20,  16,  10,   0,   9,  15,  22],  # 4 Sector D
    [ 18,  22,  14,   6,   9,   0,  11,  14],  # 5 Sector E
    [ 25,  18,  10,  12,  15,  11,   0,  10],  # 6 Sector F
    [ 30,  25,  20,  18,  22,  14,  10,   0],  # 7 Sector G
]

# ─────────────────────────────────────────────
# PARCEL METADATA  (one parcel per delivery location 1–7)
# parcel[i] = {value (₹), weight (kg), window [start_hr, end_hr]}
# ─────────────────────────────────────────────
PARCELS = [
    None,                                             # 0 Warehouse — no parcel
    {"id": 1, "loc": 1, "value": 500,  "weight": 3, "window": [9,  12]},
    {"id": 2, "loc": 2, "value": 300,  "weight": 5, "window": [10, 14]},
    {"id": 3, "loc": 3, "value": 800,  "weight": 2, "window": [9,  11]},
    {"id": 4, "loc": 4, "value": 200,  "weight": 7, "window": [12, 16]},
    {"id": 5, "loc": 5, "value": 650,  "weight": 4, "window": [11, 15]},
    {"id": 6, "loc": 6, "value": 420,  "weight": 6, "window": [10, 13]},
    {"id": 7, "loc": 7, "value": 750,  "weight": 3, "window": [9,  17]},
]

# ─────────────────────────────────────────────
# VEHICLE CONSTRAINT
# ─────────────────────────────────────────────
VEHICLE_CAPACITY = 15  # kg
VEHICLE_SPEED    = 30  # km/h  (used for arrival time estimation)
START_HOUR       = 9   # delivery shift starts at 9:00 AM

print(f"Locations : {LOCATIONS}")
print(f"Num nodes : {N}")
print(f"Vehicle   : capacity={VEHICLE_CAPACITY} kg, speed={VEHICLE_SPEED} km/h")
print("\nParcel details:")
for p in PARCELS[1:]:
    print(f"  Parcel {p['id']} → {LOCATIONS[p['loc']]:10s}  "
          f"value=₹{p['value']:4d}  weight={p['weight']} kg  "
          f"window={p['window'][0]:02d}:00–{p['window'][1]:02d}:00")

## Task 3 — Recurrence-Based Route Cost Function

We model the route cost recursively:
- **Base case**: all nodes visited → return to warehouse (cost = dist back to 0)
- **Recursive case**: try each unvisited node, take the minimum total cost

This is essentially the recursive formulation of TSP without memoization (used here for demonstration).

In [ ]:
def recursive_route_cost(current, visited, dist_matrix, nodes):
    """
    Recursively compute minimum cost to visit all unvisited nodes
    and return to warehouse (node 0).

    Parameters
    ----------
    current     : int   — current location index
    visited     : set   — set of already-visited node indices
    dist_matrix : list  — NxN distance matrix
    nodes       : list  — list of node indices to visit (excludes depot)

    Returns
    -------
    (min_cost, best_path)
    """
    unvisited = [n for n in nodes if n not in visited]

    # ── Base case: all delivery nodes visited → return to depot ──
    if not unvisited:
        return dist_matrix[current][0], [0]   # cost + return step

    # ── Recursive case: try each unvisited next node ──
    best_cost = float('inf')
    best_path = []

    for nxt in unvisited:
        travel      = dist_matrix[current][nxt]
        sub_cost, sub_path = recursive_route_cost(
            nxt, visited | {nxt}, dist_matrix, nodes
        )
        total = travel + sub_cost
        if total < best_cost:
            best_cost = total
            best_path = [nxt] + sub_path

    return best_cost, best_path


# ── Demo on small subset (nodes 1–4) to keep recursion fast ──
demo_nodes = [1, 2, 3, 4]
start = time.time()
rec_cost, rec_path = recursive_route_cost(0, set(), DIST, demo_nodes)
rec_time = time.time() - start

full_path = [0] + [p for p in rec_path if p != 0] + [0]
print("=== Recurrence-Based Route Cost ===")
print(f"Nodes visited : {[LOCATIONS[n] for n in demo_nodes]}")
print(f"Optimal path  : {' → '.join(LOCATIONS[n] for n in full_path)}")
print(f"Total cost    : {rec_cost} km")
print(f"Compute time  : {rec_time*1000:.2f} ms")

## Task 4 — Greedy + DP for Delivery Planning

### 4a. Greedy Parcel Selection (value-to-weight ratio)
Select parcels that maximise value within the vehicle's weight capacity.

In [ ]:
def greedy_parcel_selection(parcels, capacity):
    """
    Greedy fractional-knapsack style selection.
    Sort by value/weight ratio (descending), pick greedily.
    Returns list of selected parcel dicts.
    """
    candidates = [p for p in parcels if p is not None]
    # Sort by value-to-weight ratio (descending)
    candidates.sort(key=lambda p: p['value'] / p['weight'], reverse=True)

    selected   = []
    total_wt   = 0
    total_val  = 0

    for p in candidates:
        if total_wt + p['weight'] <= capacity:
            selected.append(p)
            total_wt  += p['weight']
            total_val += p['value']

    return selected, total_wt, total_val


selected_parcels, used_weight, total_value = greedy_parcel_selection(
    PARCELS, VEHICLE_CAPACITY
)

print("=== Greedy Parcel Selection ===")
print(f"Vehicle capacity : {VEHICLE_CAPACITY} kg")
print(f"Parcels selected : {len(selected_parcels)}")
print(f"Total weight     : {used_weight} kg")
print(f"Total value      : ₹{total_value}")
print("\nSelected parcels:")
print(f"  {'ID':>3}  {'Location':12}  {'Value':>6}  {'Weight':>6}  {'V/W Ratio':>9}")
print("  " + "-"*48)
for p in selected_parcels:
    ratio = p['value'] / p['weight']
    print(f"  {p['id']:>3}  {LOCATIONS[p['loc']]:12}  "
          f"₹{p['value']:>5}  {p['weight']:>5}kg  {ratio:>8.1f}")

### 4b. DP for Time-Window Validation

We check whether each selected parcel can be delivered within its time window,
given the cumulative travel time from the warehouse.

In [ ]:
def dp_time_window_check(selected, dist_matrix, speed, start_hour):
    """
    DP over delivery sequence.
    State: dp[i] = earliest feasible arrival time at location i
           coming from warehouse (node 0) visiting parcels in order.

    Returns list of (parcel, arrival_hour, feasible) tuples.
    """
    results = []
    current_loc  = 0          # start at warehouse
    current_time = start_hour # hours

    for p in selected:
        dest         = p['loc']
        travel_time  = dist_matrix[current_loc][dest] / speed  # hours
        arrival      = current_time + travel_time
        win_start, win_end = p['window']

        # Wait if arrived too early
        actual_delivery = max(arrival, win_start)
        feasible        = actual_delivery <= win_end

        results.append({
            "parcel"  : p,
            "arrival" : round(arrival, 2),
            "delivery": round(actual_delivery, 2),
            "feasible": feasible
        })

        # Update state
        current_loc  = dest
        current_time = actual_delivery + 0.25  # 15-min service time

    return results


tw_results = dp_time_window_check(
    selected_parcels, DIST, VEHICLE_SPEED, START_HOUR
)

print("=== DP Time-Window Validation ===")
print(f"  {'Parcel':>6}  {'Location':12}  {'Arrival':>7}  "
      f"{'Delivery':>8}  {'Window':>11}  {'Status':>8}")
print("  " + "-"*65)
for r in tw_results:
    p   = r['parcel']
    win = f"{p['window'][0]:02d}:00–{p['window'][1]:02d}:00"
    ok  = "✅ OK" if r['feasible'] else "❌ LATE"
    arr_h = int(r['arrival'])
    arr_m = int((r['arrival'] % 1) * 60)
    del_h = int(r['delivery'])
    del_m = int((r['delivery'] % 1) * 60)
    print(f"  {p['id']:>6}  {LOCATIONS[p['loc']]:12}  "
          f"{arr_h:02d}:{arr_m:02d}    "
          f"{del_h:02d}:{del_m:02d}     "
          f"{win:>11}  {ok:>8}")

## Task 5 — Graph Algorithms for Route Building

### 5a. Dijkstra's Algorithm — Shortest Path from Warehouse

In [ ]:
def dijkstra(dist_matrix, source, n):
    """
    Dijkstra's shortest-path algorithm from `source`.
    Returns (distances[], predecessors[]).
    """
    dist = [float('inf')] * n
    prev = [-1] * n
    dist[source] = 0
    pq = [(0, source)]   # (cost, node)

    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v in range(n):
            if u == v or dist_matrix[u][v] == 0:
                continue
            nd = dist[u] + dist_matrix[u][v]
            if nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))

    return dist, prev


def reconstruct_path(prev, source, target):
    path = []
    cur  = target
    while cur != -1:
        path.append(cur)
        cur = prev[cur]
    path.reverse()
    return path if path[0] == source else []


dijk_start = time.time()
dijk_dist, dijk_prev = dijkstra(DIST, 0, N)
dijk_time = time.time() - dijk_start

print("=== Dijkstra: Shortest Paths from Warehouse ===")
print(f"  {'Destination':12}  {'Dist (km)':>9}  {'Path'}")
print("  " + "-"*50)
for i in range(1, N):
    path = reconstruct_path(dijk_prev, 0, i)
    path_str = " → ".join(LOCATIONS[p] for p in path)
    print(f"  {LOCATIONS[i]:12}  {dijk_dist[i]:>9} km  {path_str}")
print(f"\nCompute time: {dijk_time*1000:.3f} ms")

### 5b. Minimum Spanning Tree (Prim's Algorithm)

In [ ]:
def prim_mst(dist_matrix, n):
    """
    Prim's algorithm to build an MST.
    Returns (total_cost, list_of_edges [(u,v,w)])
    """
    in_mst  = [False] * n
    key     = [float('inf')] * n
    parent  = [-1] * n
    key[0]  = 0
    pq      = [(0, 0)]

    while pq:
        _, u = heapq.heappop(pq)
        if in_mst[u]:
            continue
        in_mst[u] = True

        for v in range(n):
            if not in_mst[v] and dist_matrix[u][v] > 0:
                if dist_matrix[u][v] < key[v]:
                    key[v]    = dist_matrix[u][v]
                    parent[v] = u
                    heapq.heappush(pq, (key[v], v))

    edges      = [(parent[v], v, dist_matrix[parent[v]][v])
                  for v in range(1, n)]
    total_cost = sum(w for _, _, w in edges)
    return total_cost, edges


mst_start = time.time()
mst_cost, mst_edges = prim_mst(DIST, N)
mst_time = time.time() - mst_start

print("=== Prim's MST ===")
print(f"  {'Edge':30}  {'Weight (km)':>10}")
print("  " + "-"*43)
for u, v, w in mst_edges:
    edge_str = f"{LOCATIONS[u]} → {LOCATIONS[v]}"
    print(f"  {edge_str:30}  {w:>10} km")
print(f"\n  Total MST cost : {mst_cost} km")
print(f"  Compute time   : {mst_time*1000:.3f} ms")

## Task 6 — TSP Implementation (Held-Karp DP)

We solve TSP exactly using the **Held-Karp** algorithm (bitmask DP).
- Time complexity: O(n² · 2ⁿ)  — feasible for n ≤ 10
- We solve over the **selected delivery nodes** + depot

In [ ]:
def held_karp_tsp(dist_matrix, nodes):
    """
    Held-Karp exact TSP for a subset of nodes.
    Starts and ends at node index 0 (depot is first in `nodes`).

    Parameters
    ----------
    dist_matrix : full NxN distance matrix
    nodes       : list of node indices to visit (first element = depot)

    Returns
    -------
    (min_cost, optimal_path_as_node_indices)
    """
    n   = len(nodes)
    idx = {v: i for i, v in enumerate(nodes)}   # original index → position

    # dp[mask][i] = min cost to reach node i (position), visited = mask
    INF = float('inf')
    dp  = [[INF] * n for _ in range(1 << n)]
    par = [[-1]  * n for _ in range(1 << n)]

    dp[1][0] = 0   # start at depot (position 0), only depot visited

    for mask in range(1 << n):
        for u in range(n):
            if not (mask >> u & 1) or dp[mask][u] == INF:
                continue
            for v in range(n):
                if mask >> v & 1:
                    continue
                new_mask = mask | (1 << v)
                cost     = dp[mask][u] + dist_matrix[nodes[u]][nodes[v]]
                if cost < dp[new_mask][v]:
                    dp[new_mask][v]  = cost
                    par[new_mask][v] = u

    # All visited → return to depot
    full_mask  = (1 << n) - 1
    best_cost  = INF
    last_node  = -1
    for u in range(1, n):
        cost = dp[full_mask][u] + dist_matrix[nodes[u]][nodes[0]]
        if cost < best_cost:
            best_cost = cost
            last_node = u

    # Reconstruct path
    path_pos  = []
    mask      = full_mask
    cur       = last_node
    while cur != -1:
        path_pos.append(cur)
        prev = par[mask][cur]
        mask ^= (1 << cur)
        cur  = prev
    path_pos.reverse()
    path_nodes = [nodes[p] for p in path_pos] + [nodes[0]]  # return to depot

    return best_cost, path_nodes


# TSP over selected parcels + depot
tsp_nodes = [0] + [p['loc'] for p in selected_parcels]

tsp_start = time.time()
tsp_cost, tsp_path = held_karp_tsp(DIST, tsp_nodes)
tsp_time = time.time() - tsp_start

print("=== Held-Karp TSP ===")
print(f"Nodes in tour : {[LOCATIONS[n] for n in tsp_nodes]}")
print(f"Optimal route : {' → '.join(LOCATIONS[n] for n in tsp_path)}")
print(f"Total distance: {tsp_cost} km")
print(f"Compute time  : {tsp_time*1000:.3f} ms")

## Task 7 — Profiling & Visualization

### 7a. Runtime Profiling — TSP vs Brute-Force Scaling

In [ ]:
def brute_force_tsp(dist_matrix, nodes):
    """Brute-force TSP — tries all permutations."""
    delivery = nodes[1:]
    best_cost = float('inf')
    best_path = []
    for perm in itertools.permutations(delivery):
        route = [nodes[0]] + list(perm) + [nodes[0]]
        cost  = sum(dist_matrix[route[i]][route[i+1]] for i in range(len(route)-1))
        if cost < best_cost:
            best_cost = cost
            best_path = route
    return best_cost, best_path


# ── Scaling experiment: time both approaches for n = 3..8 ──
sizes    = list(range(3, 9))
hk_times = []
bf_times = []

# Build a consistent pool of 8 test nodes
test_pool = list(range(N))

for n in sizes:
    test_nodes = test_pool[:n]

    t0 = time.perf_counter()
    held_karp_tsp(DIST, test_nodes)
    hk_times.append((time.perf_counter() - t0) * 1000)

    t0 = time.perf_counter()
    brute_force_tsp(DIST, test_nodes)
    bf_times.append((time.perf_counter() - t0) * 1000)

print("Runtime Comparison (ms):")
print(f"  {'n':>3}  {'Held-Karp':>12}  {'Brute-Force':>12}")
print("  " + "-"*30)
for n, hk, bf in zip(sizes, hk_times, bf_times):
    print(f"  {n:>3}  {hk:>11.3f}  {bf:>11.3f}")

### 7b. Memory Profiling — Greedy & DP

In [ ]:
def run_greedy():
    greedy_parcel_selection(PARCELS, VEHICLE_CAPACITY)

def run_dp():
    dp_time_window_check(selected_parcels, DIST, VEHICLE_SPEED, START_HOUR)

mem_greedy = max(memory_usage(run_greedy))
mem_dp     = max(memory_usage(run_dp))
mem_hk     = max(memory_usage((held_karp_tsp, [DIST, tsp_nodes])))

print(f"Peak memory — Greedy  : {mem_greedy:.2f} MiB")
print(f"Peak memory — DP TW   : {mem_dp:.2f} MiB")
print(f"Peak memory — Held-Karp: {mem_hk:.2f} MiB")

### 7c. Visualization Plots

In [ ]:
# ─── Node positions for plotting ───
POS = {
    0: (5, 5),   # Warehouse (center)
    1: (2, 8),   # Sector A
    2: (4, 9),   # Sector B
    3: (7, 9),   # Sector C
    4: (9, 7),   # Sector D
    5: (9, 3),   # Sector E
    6: (6, 1),   # Sector F
    7: (2, 2),   # Sector G
}

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle("Delivery Route Optimization — Analysis Dashboard",
             fontsize=16, fontweight='bold', y=0.98)
fig.patch.set_facecolor('#1a1a2e')
for ax in axes.flat:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#0f3460')

# ─────────────── Plot 1: TSP Route Map ───────────────
ax1 = axes[0, 0]
G   = nx.Graph()
for i in range(N):
    G.add_node(i)
for i in range(N):
    for j in range(i+1, N):
        G.add_edge(i, j, weight=DIST[i][j])

node_colors = ['#e94560' if n == 0 else
               ('#4ecdc4' if n in [p['loc'] for p in selected_parcels]
                else '#555577') for n in range(N)]

nx.draw_networkx_nodes(G, POS, ax=ax1, node_color=node_colors,
                       node_size=600, alpha=0.9)
nx.draw_networkx_labels(G, POS, ax=ax1,
                        labels={i: LOCATIONS[i].replace('Sector ', 'S') for i in range(N)},
                        font_color='white', font_size=8)

# Draw all edges faintly
nx.draw_networkx_edges(G, POS, ax=ax1, alpha=0.1, edge_color='#888888')

# Draw TSP route
tsp_edges = [(tsp_path[i], tsp_path[i+1]) for i in range(len(tsp_path)-1)]
nx.draw_networkx_edges(G, POS, edgelist=tsp_edges, ax=ax1,
                       edge_color='#e94560', width=2.5, arrows=False)

ax1.set_title(f'TSP Optimal Route (Total: {tsp_cost} km)',
              color='white', fontsize=12, pad=10)
ax1.axis('off')
red_patch   = mpatches.Patch(color='#e94560', label='Depot')
teal_patch  = mpatches.Patch(color='#4ecdc4', label='Selected')
grey_patch  = mpatches.Patch(color='#555577', label='Skipped')
ax1.legend(handles=[red_patch, teal_patch, grey_patch],
           loc='lower right', facecolor='#1a1a2e', labelcolor='white', fontsize=8)

# ─────────────── Plot 2: Profit vs Weight ───────────────
ax2 = axes[0, 1]
all_parcels = [p for p in PARCELS if p is not None]
sel_ids     = {p['id'] for p in selected_parcels}
weights     = [p['weight'] for p in all_parcels]
values      = [p['value']  for p in all_parcels]
colors_sc   = ['#4ecdc4' if p['id'] in sel_ids else '#e94560' for p in all_parcels]

sc = ax2.scatter(weights, values, c=colors_sc, s=120, zorder=5, edgecolors='white', linewidths=0.5)
for p in all_parcels:
    ax2.annotate(f"P{p['id']}", (p['weight'], p['value']),
                 textcoords='offset points', xytext=(6, 4),
                 fontsize=8, color='white')
ax2.set_xlabel('Weight (kg)', color='white')
ax2.set_ylabel('Value (₹)', color='white')
ax2.set_title('Value vs Weight per Parcel', color='white', fontsize=12)
ax2.tick_params(colors='white')
ax2.axvline(VEHICLE_CAPACITY, color='#f5a623', linestyle='--',
            label=f'Capacity ({VEHICLE_CAPACITY} kg)', alpha=0.7)
sel_p  = mpatches.Patch(color='#4ecdc4', label='Selected')
skip_p = mpatches.Patch(color='#e94560', label='Skipped')
ax2.legend(handles=[sel_p, skip_p], facecolor='#1a1a2e',
           labelcolor='white', fontsize=8)
ax2.grid(alpha=0.2, color='#444466')

# ─────────────── Plot 3: Runtime Scaling ───────────────
ax3 = axes[1, 0]
ax3.plot(sizes, hk_times, 'o-', color='#4ecdc4', linewidth=2, label='Held-Karp DP', markersize=7)
ax3.plot(sizes, bf_times, 's-', color='#e94560', linewidth=2, label='Brute Force',  markersize=7)
ax3.set_xlabel('Number of Locations (n)', color='white')
ax3.set_ylabel('Time (ms)', color='white')
ax3.set_title('TSP: Held-Karp vs Brute Force Runtime', color='white', fontsize=12)
ax3.tick_params(colors='white')
ax3.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
ax3.grid(alpha=0.2, color='#444466')

# ─────────────── Plot 4: Dijkstra Distance Bar ───────────────
ax4 = axes[1, 1]
dest_labels = [LOCATIONS[i] for i in range(1, N)]
dest_dists  = [dijk_dist[i] for i in range(1, N)]
bar_colors  = ['#4ecdc4' if i+1 in sel_ids else '#555577'
               for i in range(len(dest_labels))]
bars = ax4.bar(dest_labels, dest_dists, color=bar_colors, edgecolor='white', linewidth=0.5)
for bar, d in zip(bars, dest_dists):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{d}km', ha='center', va='bottom', fontsize=8, color='white')
ax4.set_xlabel('Destination', color='white')
ax4.set_ylabel('Shortest Distance from Warehouse (km)', color='white')
ax4.set_title('Dijkstra: Shortest Distances from Warehouse', color='white', fontsize=12)
ax4.tick_params(colors='white')
ax4.grid(axis='y', alpha=0.2, color='#444466')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('images/dashboard.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Dashboard saved to images/dashboard.png")

### 7d. MST Visualization

In [ ]:
fig2, ax = plt.subplots(figsize=(8, 7))
fig2.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

G_mst = nx.Graph()
for u, v, w in mst_edges:
    G_mst.add_edge(u, v, weight=w)

mst_nc = ['#e94560' if n == 0 else '#4ecdc4' for n in G_mst.nodes()]
nx.draw_networkx_nodes(G_mst, POS, ax=ax, node_color=mst_nc, node_size=700)
nx.draw_networkx_labels(G_mst, POS, ax=ax,
                        labels={i: LOCATIONS[i].replace('Sector ', 'S') for i in G_mst.nodes()},
                        font_color='white', font_size=9)
nx.draw_networkx_edges(G_mst, POS, ax=ax, edge_color='#f5a623', width=2.5)
edge_labels = {(u, v): f"{w}km" for u, v, w in mst_edges}
nx.draw_networkx_edge_labels(G_mst, POS, edge_labels=edge_labels, ax=ax,
                              font_color='#f5a623', font_size=8)
ax.set_title(f"Prim's MST (Total cost: {mst_cost} km)",
             color='white', fontsize=13, pad=12)
ax.axis('off')
plt.tight_layout()
plt.savefig('images/mst.png', dpi=150, bbox_inches='tight',
            facecolor=fig2.get_facecolor())
plt.show()
print("✅ MST plot saved to images/mst.png")

## Task 8 — Summary Table & Analysis

### Algorithm Performance Summary

In [ ]:
print("="*70)
print(" ALGORITHM PERFORMANCE SUMMARY")
print("="*70)
print(f"{'Algorithm':<22} {'Time Complexity':<20} {'Output':<20} {'Time (ms)'}")
print("-"*70)
rows = [
    ("Recursive Route Cost", "O(n!)",         f"{rec_cost} km",    f"{rec_time*1000:.2f}"),
    ("Greedy Selection",     "O(n log n)",     f"{len(selected_parcels)} parcels", "< 0.1"),
    ("DP Time Windows",      "O(n)",           "Feasibility check", "< 0.1"),
    ("Dijkstra",             "O((V+E) log V)", "Shortest paths",    f"{dijk_time*1000:.3f}"),
    ("Prim's MST",           "O(E log V)",     f"{mst_cost} km",    f"{mst_time*1000:.3f}"),
    ("Held-Karp TSP",        "O(n² · 2ⁿ)",    f"{tsp_cost} km",    f"{tsp_time*1000:.3f}"),
]
for r in rows:
    print(f"{r[0]:<22} {r[1]:<20} {r[2]:<20} {r[3]}")
print("="*70)

print("\n📦 Final Delivery Plan:")
print(f"  Route    : {' → '.join(LOCATIONS[n] for n in tsp_path)}")
print(f"  Distance : {tsp_cost} km")
print(f"  Parcels  : {len(selected_parcels)} delivered")
print(f"  Value    : ₹{total_value}")
print(f"  Weight   : {used_weight}/{VEHICLE_CAPACITY} kg")

## Reflection & Discussion

### 1. Optimization vs Realism
The TSP route gives the globally optimal travel sequence for selected nodes, but ignores time-window ordering. The DP time-window check validates whether the greedy-selected parcel order is feasible — in practice, you'd need to combine both constraints (constrained TSP = Vehicle Routing Problem).

### 2. Algorithm Trade-offs
- **Greedy** was fastest (O(n log n)) and practically instant — good for real-time selection.
- **Dijkstra & Prim** scaled well — O(E log V) is manageable for large city graphs.
- **Held-Karp TSP** showed exponential growth. At n=8 it was still fast, but n≥15 becomes impractical.

### 3. NP-hard Challenge
TSP brute-force became noticeably slower at n=8 (8! = 40,320 permutations). Held-Karp's DP approach delays this wall significantly but still becomes infeasible around n=20–25 on typical hardware.

### 4. Visualization Insight
The value-vs-weight scatter clearly explains why some high-value parcels are skipped — their weight saturates capacity. The runtime chart dramatically shows exponential vs polynomial growth.

### 5. Feature Suggestions
- **Multi-vehicle routing** (VRP) would more realistically model a fleet.
- **Fuel cost as a function of load** would add a non-linear dimension.
- **Traffic-aware edge weights** (time-of-day DIST matrices) would make Dijkstra more realistic.